# Лекция 01: Что такое искусственный интеллект
**Курс «Основы искусственного интеллекта» · Вводная лекция**

Этот ноутбук — живое дополнение к лекции. Все код-ячейки можно запускать прямо здесь
(`Shift+Enter`), менять тексты писем и параметров модели и сразу видеть результат.

## План лекции
1. Что такое ИИ: определения и уровни (ANI / AGI / ASI)
2. Краткая история: от Тьюринга до больших языковых моделей
3. Как машины «учатся»: обучение с учителем + **интерактивный фильтр спама (3.3)**
4. Большие языковые модели: токены, предсказание следующего слова, temperature
5. Практика: **визуализация наивного байесовского классификатора**
6. Профессии в сфере ИИ: рынок труда в Китае и России (**усиленный раздел**)
7. Ограничения ИИ: галлюцинации, смещения, этика (**расширенный раздел**)
8. Итоги и задания для самостоятельной работы

> 📌 **Как работать с ноутбуком:** начните с ячейки установки зависимостей ниже,
> затем проверяйте GPU и запускайте ячейки по порядку.

## 0. Подготовка окружения
Установка зависимостей и проверка доступности GPU.

In [ ]:
# Установка зависимостей (запустите эту ячейку первой)
# В Google Colab нужные библиотеки отсутствуют по умолчанию, поэтому ставим их явно.
!pip install numpy pandas matplotlib scikit-learn seaborn wordcloud tqdm

In [ ]:
# Проверка GPU: убеждаемся, что ускоритель подключён
try:
    import torch
    print("PyTorch версия:", torch.__version__)
    print("GPU доступен:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Имя устройства:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch не установлен. Для этой лекции достаточно CPU (numpy/scikit-learn).")

In [ ]:
# Импорт остальных библиотек и общие настройки
import numpy as np
import pandas as pd
import re
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from collections import defaultdict
import os

np.random.seed(42)
sns.set_theme(style="whitegrid", font_scale=1.05)

# Стандартная рабочая директория Google Colab — /content/.
# Все сгенерированные файлы сохраняем именно туда.
SAVE_DIR = "/content" if os.path.isdir("/content") else "."
print("Файлы будут сохраняться в:", SAVE_DIR)

## 1. Что такое ИИ

**Искусственный интеллект (ИИ)** — область информатики, цель которой — создать системы,
решающие задачи, традиционно требующие человеческого интеллекта: распознавание речи
и образов, принятие решений, перевод, генерация текста, игра в стратегические игры.

### Три уровня ИИ
| Уровень | Название | Описание | Пример |
|---|---|---|---|
| **ANI** | Узкий ИИ | Мастерство в одной задаче | Фильтр спама, рекомендация товаров, AlphaGo |
| **AGI** | Общий ИИ | Интеллект уровня человека в любой задаче | Пока не создан (цель исследований) |
| **ASI** | Сверхинтеллект | Превосходит человека во всём | Гипотетический |

Сегодня мы со всех сторон окружены ANI: голосовые ассистенты **Алиса** (Яндекс) и
**Сири** (Apple), рекомендации в TikTok/抖音, распознавание лиц на турникетах метро,
скоростной транспорт 复兴号 с интеллектуальной системой управления.

### Интерактив: ваш ли это уровень?
Запустите ячейку ниже и ответьте на вопросы — программа подскажет, с каким уровнем ИИ
вы взаимодействуете каждый день.

In [ ]:
# Мини-опрос: где вокруг вас ИИ?
questions = [
    ("Пользуетесь ли вы навигатором, который строит маршрут с учётом пробок?", "традиционный ИИ (поиск + ML-прогноз времени)"),
    ("Встречали ли вы персональные рекомендации видео или товаров?", "машинное обучение (рекомендательные системы)"),
    ("Общаетесь ли с чат-ботами типа ChatGPT / Qwen / ГигаЧат?", "генеративный ИИ (большие языковые модели)"),
    ("Пользуетесь ли автопилотом/помощником водителя в автомобиле?", "ANI высокого уровня (компьютерное зрение + робототехника)"),
]

print("Ответьте мысленно «да/нет» на каждый вопрос:\n")
for i, (q, level) in enumerate(questions, 1):
    print(f"{i}. {q}")
    print(f"   -> если «да», вы уже используете: {level}")

yes = int(input("\nСколько раз вы ответили «да»? (0-4): ") or 4)
if yes >= 3:
    print("Вы ежедневно пользуетесь системами узкого ИИ (ANI). Добро пожаловать в эпоху ИИ!")
else:
    print("Присмотритесь: скорее всего, ИИ встречается вам чаще, чем кажется.")

## 2. Краткая история ИИ

- **1950** — Алан Тьюринг публикует «Computing Machinery and Intelligence», предлагает тест Тьюринга.
- **1956** — Дартмутский воркшоп: Джон Маккарти вводит термин *Artificial Intelligence*.
- **1958** — Фрэнк Розенблатт создаёт перцептрон — прообраз нейросетей.
- **1966–1973 / 1987–1993** — две «зимы ИИ»: финансирование падает, обещания не выполнены.
- **1997** — Deep Blue IBM обыгрывает Гарри Каспарова в шахматы.
- **2006** — Джеффри Хинтон: глубокое обучение (deep learning) получает новый импульс.
- **2012** — AlexNet совершает прорыв в распознавании изображений (ImageNet).
- **2016** — AlphaGo (DeepMind) побеждает Ли Седоля в го; в Китае начинается бум ИИ-стартапов
  (SenseTime, Megvii, iFLYTEK).
- **2017** — статья «Attention Is All You Need» — архитектура Transformer, основа всех современных LLM.
- **2018–2020** — BERT, GPT-2/GPT-3; китайские модели ERNIE (Baidu), PLUG (Alibaba).
- **2022–2023** — ChatGPT, Claude; в Китае —的文心一言 (ERNIE Bot),通义千问 (Qwen), Kimi, 智谱清言.
- **2024–2026** — мультимодальные модели, ИИ-агенты, открытые веса (Llama, Qwen);
  Китай принимает «Временные меры по управлению генеративным ИИ» (2023) и закон о алгоритмической рекомендации.

### Интерактивная хронология

In [ ]:
events = [
    (1950, "Тест Тьюринга"), (1956, "Термин «ИИ» (Дартмут)"), (1958, "Перцептрон"),
    (1997, "Deep Blue vs Каспаров"), (2006, "Deep Learning"), (2012, "AlexNet"),
    (2016, "AlphaGo"), (2017, "Transformer"), (2020, "GPT-3"),
    (2022, "ChatGPT"), (2023, "Qwen / ERNIE Bot"), (2025, "ИИ-агенты нового поколения"),
]
fig, ax = plt.subplots(figsize=(13, 4))
years = [e[0] for e in events]
ax.scatter(years, [0]*len(years), s=140, color="#d62728", zorder=3)
for j, (y, name) in enumerate(events):
    va = "bottom" if j % 2 == 0 else "top"
    offset = 0.12 if j % 2 == 0 else -0.12
    ax.annotate(f"{y}\n{name}", (y, 0), xytext=(y, offset),
                ha="center", va=va, fontsize=9,
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.8))
ax.axhline(0, color="gray", lw=1)
ax.set_ylim(-0.6, 0.6); ax.set_yticks([])
ax.set_title("Хронология ключевых событий истории ИИ")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "lecture01_timeline.png"), dpi=150)
plt.show()

## 3. Как машины учатся

**Программирование без ИИ:** человек пишет правила вручную.
**Машинное обучение:** человек даёт данные и пример ответа — правила машина выводит сама.

### Виды обучения
1. **С учителем (supervised)** — есть размеченные примеры: спам / не спам, фото кошки / собаки.
2. **Без учителя (unsupervised)** — разметки нет, ищем структуру: кластеризация клиентов.
3. **С подкреплением (reinforcement)** — агент учится на наградах: AlphaGo, роботы, RLHF при обучении ChatGPT/Qwen.

### 3.1 Пайплайн задачи с учителем
`данные → признаки → модель → обучение → оценка → предсказание`

### 3.2 Наивный байесовский классификатор: идея
По теореме Байеса считаем вероятность класса при документе $D$:

$$P(\text{спам} \mid D) \propto P(\text{спам}) \cdot \prod_{w \in D} P(w \mid \text{спам})$$

«Наивность» — предполагаем независимость слов. Несмотря на простоту, метод отлично
работает для фильтрации спама и лежит в основе первых почтовых фильтров.

---

## 3.3 Интерактивный фильтр спама 🔧

Здесь вы — исследователь. Мы соберём классификатор из 8 писем, а затем вы сможете:
- **написать собственное письмо** и посмотреть вердикт модели;
- **дообучить модель** новыми примерами и увидеть, как меняется решение;
- **понять, что такое переобучение**, добавив слишком мало данных.

In [ ]:
# Мини-датасет: письма и метки (1 = спам, 0 = не спам)
emails = [
    ("выигрыш в лотерею миллион долларов забрать приз срочно", 1),
    ("купи дешево скидка 90 процентов только сегодня акция", 1),
    ("бесплатный бонус заработок онлайн без вложений гарантия", 1),
    ("переведи деньги на счет срочно выигрыш приз получи", 1),
    ("привет давай встретимся в субботу обсудить проект курс", 0),
    ("отчет по продажам за квартал отправил файлы на почту", 0),
    ("напоминание лекция по машинному обучению в среду аудитория", 0),
    ("папа купи мне новую клавиатуру для программирования подарок", 0),
]

def tokenize(text):
    return re.findall(r"[а-яёa-z0-9%]+", text.lower())

class NaiveBayesSpam:
    """Крошечный наивный байесовский классификатор с объяснениями."""
    def __init__(self, alpha=1.0):
        self.alpha = alpha  # сглаживание Лапласа
        self.word_counts = {0: defaultdict(int), 1: defaultdict(int)}
        self.class_docs = {0: 0, 1: 0}
        self.vocab = set()

    def fit(self, data):
        for text, label in data:
            self.class_docs[label] += 1
            for w in tokenize(text):
                self.word_counts[label][w] += 1
                self.vocab.add(w)
        return self

    def log_prob_word(self, w, label):
        total = sum(self.word_counts[label].values())
        return np.log((self.word_counts[label][w] + self.alpha) /
                      (total + self.alpha * len(self.vocab)))

    def predict_with_explanation(self, text):
        tokens = tokenize(text)
        result = {}
        for label in (0, 1):
            lp = np.log(self.class_docs[label] / sum(self.class_docs.values()))
            contrib = {}
            for w in tokens:
                d = self.log_prob_word(w, label) - self.log_prob_word(w, 1 - label)
                lp += self.log_prob_word(w, label)
                contrib[w] = d
            result[label] = (lp, contrib)
        lps = np.array([result[0][0], result[1][0]])
        lps -= lps.max()
        probs = np.exp(lps); probs /= probs.sum()
        pred = int(probs[1] > probs[0])
        top = sorted(result[pred][1].items(), key=lambda kv: abs(kv[1]), reverse=True)[:5]
        return pred, probs, top

model = NaiveBayesSpam().fit(emails)

def check(text, m=None):
    m = m or model
    pred, probs, top = m.predict_with_explanation(text)
    verdict = "🚫 СПАМ" if pred else "✅ Нормальное письмо"
    print(f"Письмо: «{text}»")
    print(f"Вердикт: {verdict}   P(спам)={probs[1]:.3f}, P(не спам)={probs[0]:.3f}")
    print("Слова-аргументы:", ", ".join(f"{w} ({d:+.2f})" for w, d in top if abs(d) > 0.05))
    print("-" * 70)

check("получи бесплатный выигрыш срочно")
check("завтра созвон по дипломной работе в десять")

### 🎮 Задание 3.3-A: станите тестировщиком
Ниже — интерактивная ячейка. **Введите своё письмо** и посмотрите, как его оценит модель.
Попробуйте «обмануть» фильтр: напишите спам без слов «выигрыш», «бесплатно», «скидка».

In [ ]:
# Интерактив: ваше собственное письмо
my_email = input("Введите текст письма (или нажмите Enter для теста): ") or "-win срочный приз лотерея"
check(my_email)

# Вопрос для размышления: модель видит только отдельные слова.
# Почему письмо «Забронируйте бесплатную поездку — это не спам!» фильтр может пометить как спам?

### 🧪 Задание 3.3-B: дообучение на лету
Добавьте новые примеры в обучающую выборку — модель научится новым словам.
Это имитация того, как почтовые сервисы (Mail.ru, Gmail, QQ邮箱) постоянно переобучаются
на жалобах пользователей «Это спам» / «Это не спам».

In [ ]:
new_examples = [
    ("invoice payment due please transfer amount account", 1),   # новый вид спама
    ("оплатите счёт за интернет ежемесячный платёж", 0),          # похож, но не спам!
]
model2 = NaiveBayesSpam().fit(emails + new_examples)

test = "оплатите счёт ежемесячный"
print("Дообученная модель на примере:", test)
check(test, m=model2)
print("Было видно, что слово «оплатите» раньше встречалось только в спаме?")
print("Новые примеры изменили его вес — так работает активное обучение.")

### 📉 Задание 3.3-C: переобучение на пальцах
Отрежьте половину обучающих данных и посмотрите, насколько нестабильным станет модель.

In [ ]:
small_data = [e for e in emails if e[1] == 1][:2]  # только 2 спам-письма!
model_small = NaiveBayesSpam(alpha=0.05).fit(small_data)
print("Модель, обученная всего на 2 спам-письмах (нормальных примеров нет вообще):")
check("привет как дела пойдем гулять", m=model_small)
print("Вывод: мало данных + слабое сглаживание -> бессмысленные решения. Это и есть переобучение.")

## 4. Большие языковые модели (LLM)

LLM (GPT, Qwen, ERNIE Bot, ГигаЧат) решают одну задачу: **предсказать следующее слово (токен)**.
Из этогоsimple умения вырастает генерация текстов, перевод, код, рассуждения.

- **Токенизация** — текст разбивается на подсловарные единицы (токены).
- **Embedding** — каждый токен становится вектором: близкие по смыслу слова близки в пространстве.
- **Temperature** — «температура» выдачи: 0 → детерминированно и скучно, 1.5 → творчески и рискованно.
- **Контекстное окно** — сколько токенов модель «видит» одновременно (у Qwen — до 128K+).

### Живая мини-модель: цепочка Маркова
Настоящую LLM мы запустить не можем, но принцип «предсказуй следующее слово по вероятностям»
воспроизведём на 30 строках за пару секунд.

In [ ]:
corpus = """
нейросеть учится на данных и предсказывает следующее слово
большая языковая модель предсказывает следующее слово по контексту
фильтр спама решает какое слово следующее спам или не спам
машинное обучение это когда компьютер учится на данных
алгоритм обучается на примерах и предсказывает ответы
языковая модель генерирует текст слово за словом
""".strip()

chain = defaultdict(list)
for line in corpus.split("\n"):
    words = line.split() + ["<END>"]
    for a, b in zip(words, words[1:]):
        chain[a].append(b)

def generate(start="нейросеть", n=12, temperature=1.0, seed=0):
    rng = np.random.default_rng(seed)
    word, out = start, [start]
    for _ in range(n):
        nxt = chain.get(word)
        if not nxt: break
        counts = defaultdict(int)
        for w in nxt: counts[w] += 1
        ws = np.array(list(counts.values()), dtype=float) ** (1 / max(temperature, 1e-6))
        word = rng.choice(list(counts.keys()), p=ws / ws.sum())
        if word == "<END>": break
        out.append(word)
    return " ".join(out)

print("=== Temperature = 0.5 (консервативно) ===")
for s in range(3): print(" •", generate(temperature=0.5, seed=s))
print("\n=== Temperature = 3.0 (хаотично) ===")
for s in range(3): print(" •", generate(temperature=3.0, seed=s))

## 5. Визуализация работы наивного байесовского классификатора

Разберём «под капотом» три шага:
1. **Вероятности слов** в каждом классе — какие слова «тянут» письмо в спам;
2. **Логарифмические вклады** конкретных слов в решение по тестовому письму;
3. **Границу решений** модели в игрушечном двумерном пространстве признаков.

In [ ]:
# 5.1 Вероятности слов P(слово | класс)
m = model  # модель из раздела 3.3
vocab = sorted(m.vocab)
rows = []
for w in vocab:
    rows.append({"слово": w,
                 "P(w|спам)": np.exp(m.log_prob_word(w, 1)),
                 "P(w|не спам)": np.exp(m.log_prob_word(w, 0))})
df = pd.DataFrame(rows).set_index("слово")
df["lift"] = df["P(w|спам)"] / (df["P(w|не спам)"] + 1e-9)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ["#d62728" if l > 1 else "#2ca02c" for l in df["lift"]]
df[["P(w|спам)", "P(w|не спам)"]].plot.barh(ax=ax, color=["#d62728", "#2ca02c"], width=0.8)
ax.set_title("Вероятность слов в спаме и нормальных письмах (красный — спам-маркер)")
ax.set_xlabel("P(слово | класс)")
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "lecture01_nb_wordprobs.png"), dpi=150); plt.show()

In [ ]:
# 5.2 Вклад каждого слова тестового письма в решение (log-odds)
test_mail = "срочно получи бесплатный выигрыш отчет проект аудитория"
pred, probs, _ = m.predict_with_explanation(test_mail)
contrib = m.predict_with_explanation(test_mail)[2]

# полный список вкладов
tokens = tokenize(test_mail)
labels = {"спам": [], "не спам": []}
for w in tokens:
    labels["спам"].append(m.log_prob_word(w, 1) - m.log_prob_word(w, 0))
labels = pd.Series(labels["спам"], index=tokens)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels.index, labels.values,
              color=["#d62728" if v > 0 else "#2ca02c" for v in labels.values])
ax.axhline(0, color="black", lw=1)
ax.set_ylabel("Вклад в log P(спам) − log P(не спам)")
ax.set_title(f"Почему модель решила: {'СПАМ' if pred else 'НЕ СПАМ'} "
             f"(P(спам)={probs[1]:.2f}). Красный — за спам, зелёный — против")
plt.xticks(rotation=20); plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "lecture01_nb_contrib.png"), dpi=150); plt.show()

In [ ]:
# 5.3 Граница решений в 2D: признаки = вхождения слов «выигрыш» и «отчет»
X = np.array([[tok.count("выигрыш"), tok.count("отчет")] for tok in
              [tokenize(t) for t, _ in emails]], dtype=float)
y = np.array([lab for _, lab in emails])

from sklearn.naive_bayes import MultinomialNB
clf = MultinomialNB(alpha=0.5).fit(X, y)

xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 200), np.linspace(-0.5, 1.5, 200))
Z = clf.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7, 6))
ax.contourf(xx, yy, Z, levels=20, cmap="RdYlGn_r", alpha=0.8)
sc = ax.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", s=120, edgecolors="k")
ax.set_xlabel("Слово «выигрыш» (частота)")
ax.set_ylabel("Слово «отчёт» (частота)")
ax.set_title("Граница решений наивного байеса: красный — спам, жёлтый — чисто")
plt.colorbar(sc, ax=ax, label="метка (1=спам)")
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "lecture01_nb_boundary.png"), dpi=150); plt.show()

### Бонус: облако слов спам-лексики

In [ ]:
try:
    from wordcloud import WordCloud
    spam_words = " ".join(w for t, l in emails if l == 1 for w in tokenize(t)) * 3
    wc = WordCloud(width=900, height=400, background_color="white",
                   colormap="Reds", prefer_horizontal=1.0).generate(spam_words)
    plt.figure(figsize=(10, 5)); plt.imshow(wc, interpolation="bilinear"); plt.axis("off")
    plt.title("Облако слов: лексика спама в нашем датасете")
    plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "lecture01_wordcloud.png"), dpi=150); plt.show()
except Exception as e:
    print("WordCloud недоступен:", e)

## 6. Профессии в сфере ИИ (усиленный раздел)

### 6.1 Карта профессий
| Роль | Что делает | Ключевые навыки | Вилка (Россия, 2025)* | Вилка (Китай, 2025)* |
|---|---|---|---|---|
| ML-инженер | Внедряет модели в продакшн | Python, MLOps, Docker/K8s | 180–450 тыс. ₽ | 30–80 тыс. ¥ (BOSS直聘) |
| Data Scientist | Анализирует данные, строит модели | SQL, статистика, pandas | 150–350 тыс. ₽ | 25–60 тыс. ¥ (拉勾网) |
| NLP-инженер | Чат-боты, LLM, RAG, тонкая настройка | Transformers, PyTorch | 200–500 тыс. ₽ | 35–90 тыс. ¥ |
| CV-инженер | Распознавание образов, видеоналитика | OpenCV, детекторы | 170–400 тыс. ₽ | 30–70 тыс. ¥ |
| Prompt-инженер / AI-продуктовик | Ставит задачи LLM, собирает продукты | критическое мышление, домен | 120–300 тыс. ₽ | 20–50 тыс. ¥ |
| MLOps / AI-инфраструктура | Кластеры GPU, мониторинг моделей | Kubernetes, CUDA, CI/CD | 250–600 тыс. ₽ | 40–100 тыс. ¥ |

\* Ориентировочные медианные диапазоны по данным hh.ru, BOSS直聘 (bosszhipin.com), 拉勾网 (lagou.com);
для Китая — крупные города (Пекин, Шэньчжэнь, Ханчжоу, Шанхай).

### 6.2 Где искать работу: платформы
- 🇷🇺 **hh.ru**, Habr Карьера, getmatch.io — вакансии ML/Data Science в России.
- 🇨🇳 **BOSS直聘 (Boss Zhipin)** — крупнейшее приложение для поиска ИИ-вакансий в Китае:
  прямой чат с нанимающим менеджером; фильтры «算法工程师» (инженер-алгоритмщик),
  «大模型工程师» (инженер LLM), «机器学习» (машинное обучение).
- 🇨🇳 **拉勾网 (Lagou)** — IT-специализированная площадка: много вакансий в Alibaba, ByteDance,
  Tencent, SenseTime, iFLYTEK.
- Полезные ключевые слова для поиска на китайских площадках: `人工智能`, `深度学习`, `自然语言处理`,
  `计算机视觉`, `推荐算法`, `大语言模型`.

### 6.3 Реальные кейсы выпускников
> ⚠️ Кейсы ниже — собирательные типовые траектории, основанные на публично описанных
> историях выпускников программ по ИИ (ШЭН/ВШЭ, МФТИ, 清华, 浙大); используйте их как ориентир карьерной карты.

**Кейс 1. Анна, NLP-инженер (Москва).** Выпускница мехмата, пришла на курсы ИИ без опыта.
Сначала делала разметку корпуса для чат-бота банка, через год — тюнингует open-source LLM
(RuLLaMA) и собирает RAG-систему для внутренней базы знаний. Зарплата выросла ×2,5 за два года.
*Урок:* вход через данные → рост через модели и продакшн.

**Кейс 2. Ли Вэй (李伟), инженер LLM (Hangzhou, Alibaba Cloud).** Магистр 浙江大学 по CS.
Полгода готовился к собеседованиям по 算法题 (LeetCode на китайском), прошёл отбор через BOSS直聘,
начал с команды внутренних инструментов, сейчас участвует в развитии экосистемы 通义千问 (Qwen):
дообучение, дистилляция, ускорение инференса. Медианная компенсация с бонусами — выше рынка
на 30–40%. *Урок:* в Китае путь в большую ИИ-компанию почти всегда начинается с сильных
алгоритмических собеседований и открытых проектов на GitHub/ModelScope.

**Кейс 3. Мария, продакт-менеджер AI-продукта (Санкт-Петербург).** Гуманитарий, прошла
вводный курс (как этот!), собрала пет-проект — телеграм-бот резюме-анализатора на API LLM.
Нанялась junior-продактом в стартап умных магазинов, через полтора года ведёт направление
компьютерного зрения. *Урок:* в ИИ есть двери не только для программистов.

**Кейс 4. Чжан Мэй (张梅), CV-инженер (Shenzhen, DJI).** Выпускница 哈工大, занималась дронами
и SLAM. Попадала на стажировку через конкурс 中国大学生服务外包大赛, осталась в компании.
Работает над системами обхода препятствий. *Урок:* отраслевой проект + соревнования = быстрый старт.

### 6.4 Интерактив: построим вашу траекторию

In [ ]:
# Мини-калькулятор карьерной траектории
skills = []
skills.append(input("Вы умеете программировать? (да/нет): ").strip().lower())
math_ok = input("Комфортно с математикой/статистикой? (да/нет): ").strip().lower()
domain = input("Есть предметная область (финансы, медицина, лингвистика, дизайн...)? Введите или '-': ")

if skills[0] == "да" and math_ok == "да":
    path = "ML/NLP/CV-инженер: углубляйтесь в PyTorch, MLOps и опубликуйте 1-2 pet-project на GitHub."
elif skills[0] == "да":
    path = "Инженер данных / MLOps: сильный Python + SQL + инфраструктура востребованы даже без «олимпа» математики."
elif domain != "-":
    path = f"AI-продакт / аналитик в области «{domain}»: домен + понимание возможностей LLM — редкое сочетание."
else:
    path = "Начните с base: Python + табличный анализ (pandas), затем возвращайтесь к моделям."
print("\nРекомендованный трек:", path)
print("Помощь: hh.ru / BOSS直聘 / 拉勾网 — смотрите требования вакансий и закрывайте разрывы по одному навыку за месяц.")

## 7. Ограничения ИИ (расширенный раздел)

### 7.1 Галлюцинации языковых моделей
LLM оптимизированы быть *правдоподобными*, а не *истинными*. Когда знаний не хватает,
модель не признаётся, а уверенно «дорисовывает» ответ. Классические публичные примеры:

- **Юридические ссылки-призраки (США, 2023).** Адвокат Стивен Шварц использовал ChatGPT для
  подготовки иска против авиакомпании; шесть «прецедентов» оказались выдуманными
  (*Mata v. Avianca*). Суд оштрафовал юристов. Модель придумала названия дел, цитаты и номера судов.
- **«Биография» исследователя.** Пользователи просили ChatGPT написать биографию малоизвестного
  учёного — модель присваивала ему несуществующие премии и книги, сохраняя абсолютно
  уверенный тон.
- **Неверные цитаты в китайских СМИ.** ERNIE Bot и другие модели выдавали ссылки на статьи,
  которых нет; в 2024 году китайские регуляторы (CAC) потребовали от платформ маркировки
  ИИ-контента после серии фейковых новостей, сгенерированных моделью.
- **Ошибка «сколько букв r в слове strawberry»** — долгое время все LLM считали неверно:
  токенный подход ломает посимвольные операции.
- **Нестыковка арифметики.** Модель может правильно рассуждать словами, но ошибаться
  в точном подсчёте — она «помнит» паттерны, а не считает.

**Почему это происходит?** 1) objective «следующий токен» не штрафует за ложь;
2) знания «заморожены» в весах на момент обучения; 3) у модели нет доступа к фактам в момент ответа
(лечатся поиском и RAG); 4) отказ от ответа тоже обучаемое поведение, и его баланс хрупкий.

**Что делать пользователю:** проверять факты по первоисточникам, просить ссылки и перепроверять их,
использовать RAG/поиск, снижать temperature для фактических задач, критически относиться к
уверенному тону.

### 7.2 Смещения (bias)
Модели наследуют стереотипы корпусов: гендерные («медсестра — она»), профессиональные,
культурные. Примеры: дискриминационные результаты систем распознавания лиц для разных этнических
групп; стереотипные ассоциации в word2vec-векторах.

### 7.3 Этика ИИ: китайская перспектива (обсуждение)
В Китае развитие ИИ регулируется пакетом документов:
- **«Временные меры по управлению генеративным ИИ»** (《生成式人工智能服务管理暂行办法», 2023):
  контент должен соответствовать «социалистическим ценностямcore», запрещена дезинформация,
  обязательна маркировка ИИ-контента, защита персональных данных.
- **Специальные нормы по алгоритмическим рекомендациям** (2022) — право пользователя отключить
  персонализацию; запрет на злоупотребление данными («大数据杀熟» — завышение цен постоянным клиентам).
- **«Глобальная инициатива по управлению ИИ»** (《全球人工智能治理倡议», 2023) и
  **决议 ООН** при участии Китая — акцент на суверенитет, безопасность и развитие как баланс.

Сравните подходы: ЕС (AI Act — риск-ориентированная классификация), США (отраслевые рамки,
NIST AI RMF), Китай (безопасность + идеологическая содержательность + ускорение отрасли),
Россия (концепция регулирования до 2028, приоритет — развитие и недискриминация).

Обсудите в парах (2 минуты):
1. Должна ли модель, обученная на корпусе страны, отражать её ценности? Где граница между
   «культурным контекстом» и «цензурой»?
2. Кто отвечает за галлюцинацию: разработчик модели, платформа или пользователь?
3. Какие правила для ИИ вы бы предложили в своём вузе/городе?

In [ ]:
# Мини-эксперимент «галлюцинация своими руками»:
# спросите мини-модель из раздела 4 о том, чего нет в корпусе.
print(generate(start="фильтр", n=15, temperature=5.0, seed=3))
print("\nЦепочка Маркова легко порождает правдоподобные, но пустые утверждения —")
print("это упрощённая модель галлюцинаций LLM: форма есть, содержания нет.")

## 8. Итоги и домашнее задание

**Главное за лекцию:**
1. ИИ сегодня — это узкий ИИ (ANI); он силён там, где есть данные и чёткая задача.
2. ML = поиск закономерностей в данных вместо ручных правил; наивный байес — простой и
   наглядный пример.
3. LLM предсказывают следующий токен; temperature управляет креативностью; отсюда и
   талант, и галлюцинации.
4. Профессий в ИИ много, войти можно с разным бэкграундом; рынки РФ и Китая открыты
   (hh.ru, BOSS直聘, 拉勾网).
5. Ответственный пользователь ИИ проверяет факты и понимает ограничения модели.

**Домашнее задание:**
1. Дообучите фильтр спама из раздела 3.3 на своих письмах (10 спам + 10 нормальных) и
   пришлите скриншот вердиктов.
2. Найдите на BOSS直聘 или hh.ru одну вакансию ML-инженера, выпишите 5 требований и
   отметьте, какие вы уже закрываете.
3. Спровоцируйте любую LLM на галлюцинацию (несуществующая книга/статья/событие),
   зафиксируйте цитату и способ проверки. Загрузите файл в `/content/` и скачайте себе.

In [ ]:
# Финальная ячейка: итог работы с ноутбуком
# Все графики уже сохранены в стандартную директорию Colab (/content/)
print("Лекция 01 завершена. Материалы и графики сохранены в:", SAVE_DIR)
print("Домашнее задание — в предыдущей markdown-ячейке. Удачи!")